# Laboratorio interactivo — Solución de ecuaciones no lineales

## El punto de retorno seguro — Montería Drone Delivery

**Métodos:** Bisección · Regla Falsa · Newton-Raphson  
**Estudiantes:** Santiago Anaya Ramos y Andrés David Cuello Acosta  
**Asignatura:** Métodos Numéricos

La ecuación general es

\[
f(x)=D\ln(x)-\frac{E}{10}e^{0.02x}.
\]

Los valores originales del equipo son \(D=9\) y \(E=6\), pero aquí **D y E son interactivos**.
También se pueden cambiar el intervalo \([a,b]\), la aproximación inicial \(x_0\) y la tolerancia.

## 0. Python mínimo

- `x = 10` guarda un valor.
- `def f(x):` define una función.
- `if` toma una decisión.
- `for` repite instrucciones.
- `return` devuelve un resultado.
- `pandas` crea tablas.
- `matplotlib` crea gráficas.
- `ipywidgets` crea controles interactivos.

Ejecuta cada celda con **Shift + Enter**.

In [ ]:
%pip -q install ipywidgets

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, Markdown, clear_output

# 1. Modelo matemático

La función es

\[
f(x)=D\ln(x)-\frac{E}{10}e^{0.02x}.
\]

Newton-Raphson necesita además la derivada:

\[
f'(x)=\frac{D}{x}-0.002E e^{0.02x}.
\]

In [ ]:
def f(x, D, E):
    return D * math.log(x) - (E / 10) * math.exp(0.02 * x)


def df(x, D, E):
    return D / x - 0.002 * E * math.exp(0.02 * x)

# 2. Controles globales

Puedes modificar:

- **D y E**: parámetros de la ecuación.
- **a y b**: intervalo de Bisección y Regla Falsa.
- **x₀**: valor inicial de Newton-Raphson.
- **Tolerancia**: precisión del criterio de parada.

Si al cambiar D o E el intervalo deja de encerrar la raíz, pulsa **Autoajustar intervalo**.

In [ ]:
D_widget = widgets.IntSlider(
    value=9, min=1, max=9, step=1,
    description="D:", continuous_update=False
)

E_widget = widgets.IntSlider(
    value=6, min=1, max=9, step=1,
    description="E:", continuous_update=False
)

a_widget = widgets.FloatText(value=200, description="a:")
b_widget = widgets.FloatText(value=250, description="b:")
x0_widget = widgets.FloatText(value=225, description="x₀:")

tol_widget = widgets.SelectionSlider(
    options=[
        ("0.1", 1e-1),
        ("0.01", 1e-2),
        ("0.001", 1e-3),
        ("0.0001", 1e-4),
        ("0.00001", 1e-5),
    ],
    value=1e-3,
    description="Tolerancia:",
    continuous_update=False,
    style={"description_width": "initial"},
)

auto_button = widgets.Button(
    description="Autoajustar intervalo",
    button_style="info",
)

auto_output = widgets.Output()


def buscar_intervalo(D, E, xmin=1.0, xmax=1000.0, muestras=10000):
    xs = np.linspace(xmin, xmax, muestras)
    cambios = []

    anterior_x = xs[0]
    anterior_f = f(anterior_x, D, E)

    for x in xs[1:]:
        actual_f = f(x, D, E)

        if anterior_f == 0 or anterior_f * actual_f < 0:
            cambios.append((anterior_x, x))

        anterior_x = x
        anterior_f = actual_f

    if not cambios:
        return None

    # Escogemos la raíz positiva más grande: el punto de retorno del escenario.
    return cambios[-1]


def autoajustar(_):
    with auto_output:
        clear_output()

        intervalo = buscar_intervalo(D_widget.value, E_widget.value)

        if intervalo is None:
            print("No encontré un cambio de signo entre 1 y 1000 segundos.")
            return

        a, b = intervalo
        centro = (a + b) / 2

        a_widget.value = max(0.1, centro - 10)
        b_widget.value = centro + 10
        x0_widget.value = centro

        print(
            f"Intervalo ajustado a "
            f"[{a_widget.value:.4f}, {b_widget.value:.4f}]"
        )


auto_button.on_click(autoajustar)

display(
    widgets.HBox([D_widget, E_widget]),
    widgets.HBox([a_widget, b_widget, x0_widget]),
    tol_widget,
    auto_button,
    auto_output,
)

## Vista previa

La siguiente gráfica permite comprobar si el intervalo actual encierra una raíz.

In [ ]:
preview_button = widgets.Button(
    description="Graficar función",
    button_style="primary",
)
preview_output = widgets.Output()


def graficar_funcion(_=None):
    with preview_output:
        clear_output(wait=True)

        D = D_widget.value
        E = E_widget.value
        a = a_widget.value
        b = b_widget.value

        if a <= 0 or b <= 0 or a >= b:
            print("Necesitas cumplir 0 < a < b.")
            return

        margen = max(10, (b - a) * 0.25)
        xmin = max(0.1, a - margen)
        xmax = b + margen

        xs = np.linspace(xmin, xmax, 1000)
        ys = D * np.log(xs) - (E / 10) * np.exp(0.02 * xs)

        plt.figure(figsize=(12, 5.5))
        plt.plot(xs, ys, label=f"f(x), D={D}, E={E}")
        plt.axhline(0, linewidth=1)
        plt.axvspan(a, b, alpha=0.10, label="[a, b]")
        plt.scatter([a, b], [f(a, D, E), f(b, D, E)], zorder=5)

        plt.title("Función de balance de energía")
        plt.xlabel("x (segundos)")
        plt.ylabel("f(x)")
        plt.grid(alpha=0.25)
        plt.legend()
        plt.show()

        fa = f(a, D, E)
        fb = f(b, D, E)

        print(f"f(a) = {fa:.6f}")
        print(f"f(b) = {fb:.6f}")
        print("Cambio de signo:", "SÍ ✓" if fa * fb < 0 else "NO ✗")


preview_button.on_click(graficar_funcion)
display(preview_button, preview_output)
graficar_funcion()

# 3. Bisección

Calcula el punto medio

\[
m=\frac{a+b}{2}
\]

y conserva la mitad donde sigue existiendo un cambio de signo.

**Idea visual:** partir el intervalo por la mitad repetidamente.

In [ ]:
def biseccion(D, E, a, b, tolerancia=0.001, max_iter=100):
    fa = f(a, D, E)
    fb = f(b, D, E)

    if fa * fb >= 0:
        raise ValueError(
            "Bisección necesita que f(a) y f(b) tengan signos diferentes."
        )

    filas = []
    anterior = None

    for i in range(1, max_iter + 1):
        m = (a + b) / 2
        fm = f(m, D, E)

        filas.append({
            "iteración": i,
            "a": a,
            "b": b,
            "m": m,
            "f(m)": fm,
            "|f(m)|": abs(fm),
            "error_x": None if anterior is None else abs(m - anterior),
            "ancho": b - a,
        })

        if abs(fm) < tolerancia:
            break

        if fa * fm < 0:
            b = m
            fb = fm
        else:
            a = m
            fa = fm

        anterior = m

    return pd.DataFrame(filas)

In [ ]:
bis_run = widgets.Button(
    description="Ejecutar Bisección",
    button_style="success",
)
bis_slider = widgets.IntSlider(
    value=1, min=1, max=1, step=1,
    description="Iteración:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="70%"),
)
bis_output = widgets.Output()
estado_bis = {"tabla": None}


def dibujar_biseccion(numero):
    tabla = estado_bis["tabla"]
    if tabla is None:
        return

    fila = tabla.iloc[numero - 1]
    D = D_widget.value
    E = E_widget.value

    a = fila["a"]
    b = fila["b"]
    m = fila["m"]

    margen = max(2, (b - a) * 0.35)
    xs = np.linspace(max(0.1, a - margen), b + margen, 800)
    ys = D * np.log(xs) - (E / 10) * np.exp(0.02 * xs)

    with bis_output:
        clear_output(wait=True)

        plt.figure(figsize=(12, 5.5))
        plt.plot(xs, ys, label="f(x)")
        plt.axhline(0, linewidth=1)
        plt.axvspan(a, b, alpha=0.12, label="Intervalo actual")
        plt.axvline(m, linestyle=":", linewidth=2, label="m")
        plt.scatter(
            [a, m, b],
            [f(a, D, E), f(m, D, E), f(b, D, E)],
            s=70,
            zorder=5,
        )

        plt.title(f"Bisección — iteración {numero}")
        plt.xlabel("x (segundos)")
        plt.ylabel("f(x)")
        plt.grid(alpha=0.25)
        plt.legend()
        plt.show()

        siguiente = "[a, m]" if f(a, D, E) * f(m, D, E) < 0 else "[m, b]"

        display(Markdown(
            f"""
**Iteración {numero}**

- \(a={a:.6f}\)
- \(m={m:.6f}\)
- \(b={b:.6f}\)
- \(f(m)={f(m, D, E):.6f}\)
- Siguiente intervalo: **{siguiente}**
"""
        ))


def ejecutar_bis(_=None):
    try:
        tabla = biseccion(
            D_widget.value,
            E_widget.value,
            a_widget.value,
            b_widget.value,
            tol_widget.value,
        )
    except Exception as exc:
        with bis_output:
            clear_output()
            print(exc)
        return

    estado_bis["tabla"] = tabla
    bis_slider.max = len(tabla)
    bis_slider.value = 1
    dibujar_biseccion(1)


def cambiar_bis(change):
    if change["name"] == "value":
        dibujar_biseccion(change["new"])


bis_run.on_click(ejecutar_bis)
bis_slider.observe(cambiar_bis, names="value")

display(bis_run, bis_slider, bis_output)
ejecutar_bis()

# 4. Regla Falsa

Une los extremos \((a,f(a))\) y \((b,f(b))\) mediante una **secante**.

La nueva aproximación es el punto donde esa recta corta el eje \(x\):

\[
x_r=\frac{a f(b)-b f(a)}{f(b)-f(a)}.
\]

**Idea visual:** la secante “predice” dónde está la raíz.

In [ ]:
def regla_falsa(D, E, a, b, tolerancia=0.001, max_iter=100):
    fa = f(a, D, E)
    fb = f(b, D, E)

    if fa * fb >= 0:
        raise ValueError(
            "Regla Falsa necesita que f(a) y f(b) tengan signos diferentes."
        )

    filas = []
    anterior = None

    for i in range(1, max_iter + 1):
        xr = (a * fb - b * fa) / (fb - fa)
        fr = f(xr, D, E)

        filas.append({
            "iteración": i,
            "a": a,
            "b": b,
            "xr": xr,
            "f(xr)": fr,
            "|f(xr)|": abs(fr),
            "error_x": None if anterior is None else abs(xr - anterior),
        })

        if abs(fr) < tolerancia:
            break

        if fa * fr < 0:
            b = xr
            fb = fr
        else:
            a = xr
            fa = fr

        anterior = xr

    return pd.DataFrame(filas)

In [ ]:
rf_run = widgets.Button(
    description="Ejecutar Regla Falsa",
    button_style="success",
)
rf_slider = widgets.IntSlider(
    value=1, min=1, max=1, step=1,
    description="Iteración:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="70%"),
)
rf_output = widgets.Output()
estado_rf = {"tabla": None}


def dibujar_regla_falsa(numero):
    tabla = estado_rf["tabla"]
    if tabla is None:
        return

    fila = tabla.iloc[numero - 1]
    D = D_widget.value
    E = E_widget.value

    a = fila["a"]
    b = fila["b"]
    xr = fila["xr"]

    fa = f(a, D, E)
    fb = f(b, D, E)

    margen = max(2, (b - a) * 0.25)
    xs = np.linspace(max(0.1, a - margen), b + margen, 900)
    ys = D * np.log(xs) - (E / 10) * np.exp(0.02 * xs)

    sec_x = np.linspace(a, b, 100)
    sec_y = fa + (fb - fa) * (sec_x - a) / (b - a)

    with rf_output:
        clear_output(wait=True)

        plt.figure(figsize=(12, 5.5))
        plt.plot(xs, ys, label="f(x)")
        plt.plot(sec_x, sec_y, linestyle="--", label="Secante")
        plt.axhline(0, linewidth=1)
        plt.axvspan(a, b, alpha=0.08, label="Intervalo actual")
        plt.scatter([a, b], [fa, fb], s=70, zorder=5)
        plt.scatter(
            [xr], [0],
            s=90, marker="x", zorder=6,
            label=f"xr = {xr:.6f}",
        )
        plt.scatter([xr], [f(xr, D, E)], s=65, zorder=6)

        plt.title(f"Regla Falsa — iteración {numero}")
        plt.xlabel("x (segundos)")
        plt.ylabel("f(x)")
        plt.grid(alpha=0.25)
        plt.legend()
        plt.show()

        display(Markdown(
            f"""
**Iteración {numero}**

- \(a={a:.6f}\)
- \(b={b:.6f}\)
- \(x_r={xr:.6f}\)
- \(f(x_r)={f(xr, D, E):.6f}\)

El corte de la secante con el eje \(x\) genera la nueva aproximación.
"""
        ))


def ejecutar_rf(_=None):
    try:
        tabla = regla_falsa(
            D_widget.value,
            E_widget.value,
            a_widget.value,
            b_widget.value,
            tol_widget.value,
        )
    except Exception as exc:
        with rf_output:
            clear_output()
            print(exc)
        return

    estado_rf["tabla"] = tabla
    rf_slider.max = len(tabla)
    rf_slider.value = 1
    dibujar_regla_falsa(1)


def cambiar_rf(change):
    if change["name"] == "value":
        dibujar_regla_falsa(change["new"])


rf_run.on_click(ejecutar_rf)
rf_slider.observe(cambiar_rf, names="value")

display(rf_run, rf_slider, rf_output)
ejecutar_rf()

# 5. Newton-Raphson

Newton parte de una aproximación \(x_n\), calcula la **tangente** en ese punto y usa
su corte con el eje \(x\) como siguiente aproximación:

\[
x_{n+1}=x_n-\frac{f(x_n)}{f'(x_n)}.
\]

**Idea visual:** la tangente “salta” hacia la raíz.

In [ ]:
def newton_raphson(D, E, x0, tolerancia=0.001, max_iter=100):
    if x0 <= 0:
        raise ValueError("x₀ debe ser positivo.")

    filas = []
    x = x0

    for i in range(1, max_iter + 1):
        fx = f(x, D, E)
        dfx = df(x, D, E)

        if abs(dfx) < 1e-12:
            raise ValueError("La derivada es demasiado cercana a cero.")

        x_nuevo = x - fx / dfx

        if x_nuevo <= 0:
            raise ValueError(
                "Newton salió del dominio x > 0. Prueba otro x₀."
            )

        filas.append({
            "iteración": i,
            "xn": x,
            "f(xn)": fx,
            "f'(xn)": dfx,
            "xn+1": x_nuevo,
            "|f(xn+1)|": abs(f(x_nuevo, D, E)),
            "error_x": abs(x_nuevo - x),
        })

        if abs(f(x_nuevo, D, E)) < tolerancia:
            break

        x = x_nuevo

    return pd.DataFrame(filas)

In [ ]:
newton_run = widgets.Button(
    description="Ejecutar Newton",
    button_style="success",
)
newton_slider = widgets.IntSlider(
    value=1, min=1, max=1, step=1,
    description="Iteración:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="70%"),
)
newton_output = widgets.Output()
estado_newton = {"tabla": None}


def dibujar_newton(numero):
    tabla = estado_newton["tabla"]
    if tabla is None:
        return

    fila = tabla.iloc[numero - 1]
    D = D_widget.value
    E = E_widget.value

    xn = fila["xn"]
    siguiente = fila["xn+1"]
    fx = f(xn, D, E)
    pendiente = df(xn, D, E)

    izquierda = max(0.1, min(xn, siguiente) - 12)
    derecha = max(xn, siguiente) + 12

    xs = np.linspace(izquierda, derecha, 900)
    ys = D * np.log(xs) - (E / 10) * np.exp(0.02 * xs)
    tangente = fx + pendiente * (xs - xn)

    with newton_output:
        clear_output(wait=True)

        plt.figure(figsize=(12, 5.5))
        plt.plot(xs, ys, label="f(x)")
        plt.plot(xs, tangente, linestyle="--", label="Tangente en xn")
        plt.axhline(0, linewidth=1)
        plt.scatter([xn], [fx], s=80, zorder=5, label=f"xn = {xn:.6f}")
        plt.scatter(
            [siguiente], [0],
            s=90, marker="x", zorder=6,
            label=f"xn+1 = {siguiente:.6f}",
        )

        plt.title(f"Newton-Raphson — iteración {numero}")
        plt.xlabel("x (segundos)")
        plt.ylabel("f(x)")
        plt.grid(alpha=0.25)
        plt.legend()
        plt.show()

        display(Markdown(
            f"""
**Iteración {numero}**

- \(x_n={xn:.6f}\)
- \(f(x_n)={fx:.6f}\)
- \(f'(x_n)={pendiente:.6f}\)
- \(x_{{n+1}}={siguiente:.6f}\)

El corte de la tangente con el eje \(x\) produce la siguiente aproximación.
"""
        ))


def ejecutar_newton(_=None):
    try:
        tabla = newton_raphson(
            D_widget.value,
            E_widget.value,
            x0_widget.value,
            tol_widget.value,
        )
    except Exception as exc:
        with newton_output:
            clear_output()
            print(exc)
        return

    estado_newton["tabla"] = tabla
    newton_slider.max = len(tabla)
    newton_slider.value = 1
    dibujar_newton(1)


def cambiar_newton(change):
    if change["name"] == "value":
        dibujar_newton(change["new"])


newton_run.on_click(ejecutar_newton)
newton_slider.observe(cambiar_newton, names="value")

display(newton_run, newton_slider, newton_output)
ejecutar_newton()

# 6. Comparación de los tres métodos

Ejecutamos los tres con la **misma función** y la **misma tolerancia**.
Así podemos comparar cantidad de iteraciones, raíz aproximada y residuo.

In [ ]:
comparar_button = widgets.Button(
    description="Comparar los 3 métodos",
    button_style="primary",
)
comparar_output = widgets.Output()


def comparar(_=None):
    with comparar_output:
        clear_output(wait=True)

        D = D_widget.value
        E = E_widget.value
        a = a_widget.value
        b = b_widget.value
        x0 = x0_widget.value
        tol = tol_widget.value

        try:
            tb = biseccion(D, E, a, b, tol)
            tr = regla_falsa(D, E, a, b, tol)
            tn = newton_raphson(D, E, x0, tol)
        except Exception as exc:
            print("No fue posible comparar:", exc)
            return

        resultados = pd.DataFrame([
            {
                "Método": "Bisección",
                "Iteraciones": len(tb),
                "Raíz aproximada": tb.iloc[-1]["m"],
                "Residuo |f(x)|": abs(tb.iloc[-1]["f(m)"]),
            },
            {
                "Método": "Regla Falsa",
                "Iteraciones": len(tr),
                "Raíz aproximada": tr.iloc[-1]["xr"],
                "Residuo |f(x)|": abs(tr.iloc[-1]["f(xr)"]),
            },
            {
                "Método": "Newton-Raphson",
                "Iteraciones": len(tn),
                "Raíz aproximada": tn.iloc[-1]["xn+1"],
                "Residuo |f(x)|": abs(f(tn.iloc[-1]["xn+1"], D, E)),
            },
        ])

        print(f"D={D}, E={E}, tolerancia={tol}")
        display(resultados.round(8))

        plt.figure(figsize=(10, 5))
        plt.bar(resultados["Método"], resultados["Iteraciones"])
        plt.title("Iteraciones necesarias")
        plt.ylabel("Iteraciones")
        plt.grid(axis="y", alpha=0.25)
        plt.show()

        plt.figure(figsize=(11, 5))
        plt.plot(tb["iteración"], tb["m"], marker="o", label="Bisección")
        plt.plot(tr["iteración"], tr["xr"], marker="o", label="Regla Falsa")
        plt.plot(
            tn["iteración"],
            tn["xn+1"],
            marker="o",
            label="Newton-Raphson",
        )

        plt.title("Cómo se acercan a la raíz")
        plt.xlabel("Iteración")
        plt.ylabel("Aproximación")
        plt.grid(alpha=0.25)
        plt.legend()
        plt.show()

        display(Markdown("### Tabla — Bisección"))
        display(tb.round(8))

        display(Markdown("### Tabla — Regla Falsa"))
        display(tr.round(8))

        display(Markdown("### Tabla — Newton-Raphson"))
        display(tn.round(8))


comparar_button.on_click(comparar)
display(comparar_button, comparar_output)
comparar()

# 7. Lectura conceptual

### Bisección
Usa el **punto medio** y prioriza la robustez.

### Regla Falsa
Usa una **secante** y aprovecha los valores de la función en los extremos.

### Newton-Raphson
Usa una **tangente**, necesita la derivada y puede converger muy rápido si \(x_0\) es adecuado.

Con los valores originales \(D=9\), \(E=6\), \([200,250]\), \(x_0=225\) y tolerancia
\(0.001\), puedes pulsar **Comparar los 3 métodos** y observar directamente la diferencia.

# 8. Guion breve para la sustentación

### Introducción
> “Nuestra ecuación depende de dos parámetros, D y E. En nuestro caso son 9 y 6, pero los dejamos interactivos para visualizar cómo cambia el problema.”

### Bisección
> “Bisección parte el intervalo a la mitad y conserva la zona donde sigue existiendo un cambio de signo.”

### Regla Falsa
> “Regla Falsa también trabaja con un intervalo, pero usa una recta secante. El corte de esa recta con el eje x produce la nueva aproximación.”

### Newton-Raphson
> “Newton utiliza la derivada para construir una tangente. El punto donde la tangente corta el eje x se convierte en la siguiente aproximación.”

### Convergencia
> “Convergencia significa que las aproximaciones se acercan progresivamente a una raíz.”

### Comparación
> “Los tres métodos buscan la misma raíz, pero utilizan información diferente: mitad del intervalo, secante o tangente.”

### Cierre
> “No existe un método universalmente mejor. La elección depende de estabilidad, información disponible y costo computacional.”

# 9. Preguntas posibles

**¿Por qué Bisección y Regla Falsa necesitan un intervalo?**  
Porque ambos conservan un intervalo donde existe un cambio de signo.

**¿Por qué Newton necesita la derivada?**  
Porque usa la pendiente de la tangente para calcular la siguiente aproximación.

**¿Qué pasa si cambia D o E?**  
Cambia la función y puede cambiar la posición de la raíz. Por eso también se pueden reajustar el intervalo y \(x_0\).

**¿Qué significa tolerancia?**  
Es la precisión exigida antes de detener el método.

**¿Newton siempre es más conveniente?**  
No. Puede ser muy rápido, pero depende del valor inicial y de que la derivada sea adecuada.

**¿Por qué x debe ser positivo?**  
Porque aparece \(\ln(x)\), que en números reales requiere \(x>0\).